<a href="https://colab.research.google.com/github/LuizFelipeSchroderMarcon/ETL_PRECO_COMBUSTIVEL/blob/main/ETL_PRECO_COMBUSTIVEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autor : Luiz Felipe Schroder Marcon
# ETL — Preços de Combustíveis ANP
# Fonte : Agência Nacional do Petróleo, Gás Natural e Biocombustíveis (ANP)
# Período: 03/05/2026 a 09/05/2026

# 1- Instalação de dependências

In [ ]:
!pip install openpyxl -q

# 2- Imports

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
from datetime import datetime

#Extract
#3- Upload do Arquivo

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving revendas_lpc_2026-05-03_2026-05-09.xlsx to revendas_lpc_2026-05-03_2026-05-09.xlsx


#4- Configuração e leitura

In [ ]:
ARQUIVO_XLSX = "revendas_lpc_2026-05-03_2026-05-09.xlsx"
HEADER_ROW   = 9

print("Extraindo dados da planilha...")
df_raw = pd.read_excel(
    ARQUIVO_XLSX,
    header=HEADER_ROW,
    dtype={"CNPJ": str, "CEP": str},
)

print(f"Registros extraídos: {len(df_raw):,}")
print(f"Colunas: {df_raw.shape[1]}")

Extraindo dados da planilha...
Extração concluída!
Registros extraídos: 19,354
Colunas: 15


#5- Diagnóstico dos dados brutos

In [ ]:
print("Diagnóstico inicial dos dados brutos:")
print(f"Total de registros: {len(df_raw):,}")
print(f"Registros duplicados: {df_raw.duplicated().sum():,}")
print("Valores nulos por coluna:")
nulls = df_raw.isnull().sum()
for col, n in nulls.items():
    pct = n / len(df_raw) * 100
    print(f"   {col:<22} {n:>6,} ({pct:5.1f}%)")
print("\n   Tipos de dados:")
print(df_raw.dtypes.to_string())

Diagnóstico inicial dos dados brutos:
Total de registros: 19,354
Registros duplicados: 0
Valores nulos por coluna:
   CNPJ                        0 (  0.0%)
   RAZÃO                       0 (  0.0%)
   FANTASIA               11,816 ( 61.1%)
   ENDEREÇO                    0 (  0.0%)
   NÚMERO                      0 (  0.0%)
   COMPLEMENTO            14,948 ( 77.2%)
   BAIRRO                     17 (  0.1%)
   CEP                         0 (  0.0%)
   MUNICÍPIO                   0 (  0.0%)
   ESTADO                      0 (  0.0%)
   BANDEIRA                    0 (  0.0%)
   PRODUTO                     0 (  0.0%)
   UNIDADE DE MEDIDA           0 (  0.0%)
   PREÇO DE REVENDA            0 (  0.0%)
   DATA DA COLETA              0 (  0.0%)

   Tipos de dados:
CNPJ                         object
RAZÃO                        object
FANTASIA                     object
ENDEREÇO                     object
NÚMERO                       object
COMPLEMENTO                  object
BAIRRO             

#TRANSFORM

In [ ]:
#Cópia do arquivo original
df = df_raw.copy()

#6- Renomear colunas

In [ ]:
COLUNAS = {
    "CNPJ": "cnpj",
    "RAZÃO": "razao_social",
    "FANTASIA": "nome_fantasia",
    "ENDEREÇO": "endereco",
    "NÚMERO": "numero",
    "COMPLEMENTO": "complemento",
    "BAIRRO": "bairro",
    "CEP": "cep",
    "MUNICÍPIO": "municipio",
    "ESTADO": "estado",
    "BANDEIRA": "bandeira",
    "PRODUTO": "produto",
    "UNIDADE DE MEDIDA": "unidade_medida",
    "PREÇO DE REVENDA": "preco_revenda",
    "DATA DA COLETA": "data_coleta",
}
df.rename(columns=COLUNAS, inplace=True)
print("Colunas renomeadas para snake_case.")

Colunas renomeadas para snake_case.


#7- Formatar CNPJ

In [ ]:
#Função para padronizar o cnpj removendo não-dígitos, preenchendo zeros a esquerda e aplicando a máscara da formatação de CNPJ
def formatar_cnpj(cnpj: str) -> str:
    if pd.isna(cnpj):
        return np.nan
    digitos = "".join(filter(str.isdigit, str(cnpj)))
    digitos = digitos.zfill(14)
    return f"{digitos[:2]}.{digitos[2:5]}.{digitos[5:8]}/{digitos[8:12]}-{digitos[12:14]}"

df["cnpj_formatado"] = df["cnpj"].apply(formatar_cnpj)

cnpj_invalidos = df["cnpj"].apply(
    lambda x: len("".join(filter(str.isdigit, str(x)))) != 14
    if pd.notna(x) else False
).sum()

print("CNPJ formatado")
print(f"CNPJs com dígitos != 14: {cnpj_invalidos}")

CNPJ formatado
CNPJs com dígitos != 14: 9775


#8- Formatar CEP

In [ ]:
#Padroniza o CEP
def formatar_cep(cep) -> str:
    if pd.isna(cep):
        return np.nan
    digitos = "".join(filter(str.isdigit, str(cep)))
    digitos = digitos.zfill(8)
    return f"{digitos[:5]}-{digitos[5:]}"

df["cep"] = df["cep"].apply(formatar_cep)
print("CEPs padronizados")

CEPs padronizados


#9- Padronizar campo NÚMERO

In [ ]:
df["numero"] = (
    df["numero"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        "S/N": "SEM NÚMERO",
        "SN": "SEM NÚMERO",
        "S.N": "SEM NÚMERO",
        "NAN": "SEM NÚMERO",
        "NONE": "SEM NÚMERO",
    })
)
print("Campo 'numero' padronizado.")
print(f"Registros 'SEM NÚMERO': {(df['numero'] == 'SEM NÚMERO').sum():,}")

Campo 'numero' padronizado.
Registros 'SEM NÚMERO': 1,993


#10- Tratar valores nulos

In [ ]:
df["nome_fantasia"] = df["nome_fantasia"].fillna(df["razao_social"])
df["complemento"] = df["complemento"].fillna("")
df["bairro"] = df["bairro"].fillna("NÃO INFORMADO")

print("Valores nulos tratados.")
print(f"nome_fantasia nulos restantes: {df['nome_fantasia'].isna().sum()}")
print(f"complemento nulos restantes: {df['complemento'].isna().sum()}")
print(f"bairro nulos restantes: {df['bairro'].isna().sum()}")

Valores nulos tratados.
nome_fantasia nulos restantes: 0
complemento nulos restantes: 0
bairro nulos restantes: 0


#11- Arredondar e validar preços

In [ ]:
df["preco_revenda"] = df["preco_revenda"].round(2)

preco_invalidos = (df["preco_revenda"] <= 0).sum()
df = df[df["preco_revenda"] > 0]

print("Preços validados e arredondados (2 casas decimais).")
print(f"Registros com preço inválido removidos: {preco_invalidos}")
print("\nEstatísticas dos preços por produto:")
resumo_preco = (
    df.groupby("produto")["preco_revenda"]
    .agg(["min", "mean", "max", "count"])
    .round(2)
    .rename(columns={"min": "Mín (R$)", "mean": "Média (R$)",
                     "max": "Máx (R$)", "count": "Registros"})
)
print(resumo_preco.to_string())

Preços validados e arredondados (2 casas decimais).
Registros com preço inválido removidos: 0

Estatísticas dos preços por produto:
                    Mín (R$)  Média (R$)  Máx (R$)  Registros
produto                                                      
DIESEL S10              5.69        7.26      9.99       2993
DIESEL S500             5.95        7.11      8.99       1773
ETANOL                  2.98        4.70      6.94       3601
GASOLINA ADITIVADA      5.79        6.90      9.79       3321
GASOLINA COMUM          5.49        6.70      9.59       4262
GLP                    79.99      115.56    156.00       3084
GNV                     3.69        4.62      6.28        320


#12- Padronizar colunas de texto

In [ ]:
cols_texto = [
    "razao_social", "nome_fantasia", "endereco",
    "bairro", "municipio", "estado", "bandeira", "produto",
]
for col in cols_texto:
    df[col] = df[col].astype(str).str.strip().str.upper()

print("Colunas de texto padronizadas")

Colunas de texto padronizadas


#13- Remover duplicatas

In [ ]:
antes = len(df)
df.drop_duplicates(
    subset=["cnpj", "produto", "data_coleta"],
    keep="last",
    inplace=True,
)
depois = len(df)
print(f"Duplicatas removidas: {antes - depois:,} registros.")
print(f"Registros restantes: {depois:,}")

Duplicatas removidas: 0 registros.
Registros restantes: 19,354


#14- Enriquecimento — colunas derivadas

In [ ]:
GRUPO_PRODUTO = {
    "GASOLINA COMUM": "GASOLINA",
    "GASOLINA ADITIVADA": "GASOLINA",
    "DIESEL S500": "DIESEL",
    "DIESEL S10": "DIESEL",
    "ETANOL": "BIOCOMBUSTÍVEL",
    "GLP": "GÁS",
    "GNV": "GÁS",
}
df["grupo_produto"] = df["produto"].map(GRUPO_PRODUTO)

SIGLAS_ESTADO = {
    "ACRE": "AC", "ALAGOAS": "AL", "AMAPÁ": "AP", "AMAZONAS": "AM",
    "BAHIA": "BA", "CEARÁ": "CE", "DISTRITO FEDERAL": "DF",
    "ESPÍRITO SANTO": "ES", "GOIÁS": "GO", "MARANHÃO": "MA",
    "MATO GROSSO": "MT", "MATO GROSSO DO SUL": "MS", "MINAS GERAIS": "MG",
    "PARÁ": "PA", "PARAÍBA": "PB", "PARANÁ": "PR", "PERNAMBUCO": "PE",
    "PIAUÍ": "PI", "RIO DE JANEIRO": "RJ", "RIO GRANDE DO NORTE": "RN",
    "RIO GRANDE DO SUL": "RS", "RONDÔNIA": "RO", "RORAIMA": "RR",
    "SANTA CATARINA": "SC", "SÃO PAULO": "SP", "SERGIPE": "SE",
    "TOCANTINS": "TO",
}
df["uf"] = df["estado"].map(SIGLAS_ESTADO)

df["data_coleta"]   = pd.to_datetime(df["data_coleta"])
df["semana_inicio"] = (
    df["data_coleta"] - pd.to_timedelta(df["data_coleta"].dt.dayofweek, unit="D")
).dt.date
df["ano_coleta"] = df["data_coleta"].dt.year
df["mes_coleta"] = df["data_coleta"].dt.month

#Classifica o preço em BAIXO/MÉDIO/ALTO
def faixa_preco(row):
    p = row["preco_revenda"]
    produto = row["produto"]
    if produto == "GLP":
        if p < 100: return "BAIXO"
        elif p <= 130: return "MÉDIO"
        else: return "ALTO"
    else:
        if p < 5.5: return "BAIXO"
        elif p <= 7.5: return "MÉDIO"
        else: return "ALTO"

df["faixa_preco"] = df.apply(faixa_preco, axis=1)

print(" Novas colunas derivadas criadas:")
print("-- grupo_produto  — Agrupamento do tipo de combustível")
print("-- uf             — Sigla do estado (UF)")
print("-- semana_inicio  — Início da semana de coleta")
print("-- ano_coleta     — Ano da coleta")
print("-- mes_coleta     — Mês da coleta")
print("-- faixa_preco    — Classificação: BAIXO / MÉDIO / ALTO")
print(f"\nDistribuição por grupo de produto:")
print(df["grupo_produto"].value_counts().to_string())

 Novas colunas derivadas criadas:
-- grupo_produto  — Agrupamento do tipo de combustível
-- uf             — Sigla do estado (UF)
-- semana_inicio  — Início da semana de coleta
-- ano_coleta     — Ano da coleta
-- mes_coleta     — Mês da coleta
-- faixa_preco    — Classificação: BAIXO / MÉDIO / ALTO

Distribuição por grupo de produto:
grupo_produto
GASOLINA          7583
DIESEL            4766
BIOCOMBUSTÍVEL    3601
GÁS               3404


#15- Reorganizar colunas

In [ ]:
COLUNAS_FINAL = [
    "cnpj", "cnpj_formatado", "razao_social", "nome_fantasia",
    "endereco", "numero", "complemento", "bairro", "cep",
    "municipio", "estado", "uf",
    "bandeira", "produto", "grupo_produto", "unidade_medida",
    "preco_revenda", "faixa_preco",
    "data_coleta", "semana_inicio", "ano_coleta", "mes_coleta",
]
df = df[COLUNAS_FINAL]

print(f"   Registros finais : {len(df):,}")
print(f"   Colunas finais   : {df.shape[1]}")
df.info()

   Registros finais : 19,354
   Colunas finais   : 22
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19354 entries, 0 to 19353
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   cnpj            19354 non-null  object        
 1   cnpj_formatado  19354 non-null  object        
 2   razao_social    19354 non-null  object        
 3   nome_fantasia   19354 non-null  object        
 4   endereco        19354 non-null  object        
 5   numero          19354 non-null  object        
 6   complemento     19354 non-null  object        
 7   bairro          19354 non-null  object        
 8   cep             19354 non-null  object        
 9   municipio       19354 non-null  object        
 10  estado          19354 non-null  object        
 11  uf              9248 non-null   object        
 12  bandeira        19354 non-null  object        
 13  produto         19354 non-null  object        
 14  

#LOAD

#16- Exportar CSV e XLSX limpos

In [ ]:
CSV_SAIDA = "combustiveis_anp_tratado.csv"

df.to_csv(CSV_SAIDA, index=False, encoding="utf-8-sig", sep=";")
df.to_excel(CSV_SAIDA.replace(".csv", ".xlsx"), index=False)

print("CSV e XLSX exportados")

CSV e XLSX exportados


#17- Download dos arquivos

In [ ]:
from google.colab import files
files.download(CSV_SAIDA)
files.download(CSV_SAIDA.replace(".csv", ".xlsx"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#18- Resumo final do pipeline

In [ ]:
print("RELATÓRIO FINAL DO PIPELINE ETL")
print("Fonte: ANP — Preços de Combustíveis")

print(f"\nArquivo origem: {ARQUIVO_XLSX}")
print(f"Registros brutos: {len(df_raw):,}")

print(f"Registros finais: {len(df):,}")
print(f"Colunas originais: {df_raw.shape[1]}")
print(f"Colunas finais: {df.shape[1]}")
print(f"Novos campos: grupo_produto, uf, semana_inicio, ano_coleta, mes_coleta, faixa_preco, cnpj_formatado")

print(f"\nEstados cobertos: {df['estado'].nunique()}")
print(f"Municípios: {df['municipio'].nunique():,}")
print(f"Postos únicos: {df['cnpj'].nunique():,}")
print(f"Produtos: {sorted(df['produto'].unique())}")

RELATÓRIO FINAL DO PIPELINE ETL
Fonte: ANP — Preços de Combustíveis

Arquivo origem: revendas_lpc_2026-05-03_2026-05-09.xlsx
Registros brutos: 19,354
Registros finais: 19,354
Colunas originais: 15
Colunas finais: 22
Novos campos: grupo_produto, uf, semana_inicio, ano_coleta, mes_coleta, faixa_preco, cnpj_formatado

Estados cobertos: 27
Municípios: 384
Postos únicos: 7,246
Produtos: ['DIESEL S10', 'DIESEL S500', 'ETANOL', 'GASOLINA ADITIVADA', 'GASOLINA COMUM', 'GLP', 'GNV']
